# pymfx — Format Conversion

This notebook demonstrates how to export a `.mfx` file to the four supported geospatial formats:

| Format | Function | Output |
|--------|----------|--------|
| GeoJSON | `pymfx.to_geojson(mfx)` | GeoJSON FeatureCollection |
| GPX | `pymfx.to_gpx(mfx)` | GPX 1.1 track |
| KML | `pymfx.to_kml(mfx)` | KML Placemark |
| CSV | `pymfx.to_csv(mfx)` | Flat CSV with one row per point |

All functions return a `str`. Use the CLI `--export` flag to write directly to a file.

**Requirements:** no extra dependencies — conversion is pure Python.

In [ ]:
import sys, math, uuid
sys.path.insert(0, '..')  # if running from notebooks/

import pymfx
from pymfx.models import (
    MfxFile, Meta, Trajectory, TrajectoryPoint, SchemaField,
    Events, Event, Index,
)
from pymfx import compute_checksum

print(f"pymfx {pymfx.__version__}")

## Build a demo mission

We create a 40-point alpine survey flight with altitude, speed, heading and a few events.

In [ ]:
BASE_LAT, BASE_LON = 45.8326, 6.8652   # Chamonix
N = 40

raw_pts = [
    (
        round(i * 0.5, 3),
        round(BASE_LAT + i * 0.00008, 6),
        round(BASE_LON + math.sin(i * 0.25) * 0.0004, 6),
        round(1800.0 + math.sin(i * 0.3) * 20, 1),
        round(9.0 + math.cos(i * 0.2) * 2, 1),
        round((i * 9) % 360, 1),
    )
    for i in range(N)
]

schema_fields = [
    SchemaField('t',        'float',   ['no_null']),
    SchemaField('lat',      'float',   ['no_null']),
    SchemaField('lon',      'float',   ['no_null']),
    SchemaField('alt_m',    'float32'),
    SchemaField('speed_ms', 'float32'),
    SchemaField('heading',  'float32'),
]

points    = [TrajectoryPoint(t=r[0], lat=r[1], lon=r[2], alt_m=r[3], speed_ms=r[4], heading=r[5]) for r in raw_pts]
raw_lines = [f"{r[0]:.3f} | {r[1]} | {r[2]} | {r[3]} | {r[4]} | {r[5]}" for r in raw_pts]

event_schema = [
    SchemaField('t',        'float', ['no_null']),
    SchemaField('type',     'str',   ['enum=[takeoff,landing,waypoint,anomaly]']),
    SchemaField('severity', 'str',   ['enum=[info,warning,critical]']),
    SchemaField('detail',   'str'),
]
event_list = [
    Event(t=0.0,  type='takeoff',  severity='info',    detail='nominal'),
    Event(t=5.0,  type='waypoint', severity='info',    detail='summit_approach'),
    Event(t=12.0, type='anomaly',  severity='warning', detail='wind_gust_12ms'),
    Event(t=19.0, type='landing',  severity='info',    detail='nominal'),
]
ev_raw = [f"{e.t:.3f} | {e.type} | {e.severity} | {e.detail}" for e in event_list]

lats = [p.lat for p in points]
lons = [p.lon for p in points]
bbox = (round(min(lons), 6), round(min(lats), 6), round(max(lons), 6), round(max(lats), 6))

mfx = MfxFile(
    version='1.0', encoding='UTF-8',
    meta=Meta(
        id=f'uuid:{uuid.uuid4()}',
        drone_id='DJI-Air3-SN556677',
        drone_type='multirotor',
        pilot_id='FR-PILOT-0099',
        date_start='2026-07-14T07:00:00Z',
        date_end='2026-07-14T07:00:20Z',
        duration_s=19.5,
        status='complete',
        application='alpine_survey',
        location='Chamonix, France',
        sensors=['rgb', 'lidar'],
        data_level='raw',
        license='CC-BY-4.0',
        contact='survey@alpinelab.fr',
    ),
    trajectory=Trajectory(
        frequency_hz=2.0,
        schema_fields=schema_fields,
        points=points,
        checksum=compute_checksum(raw_lines),
        raw_lines=raw_lines,
    ),
    events=Events(
        schema_fields=event_schema,
        events=event_list,
        checksum=compute_checksum(ev_raw),
        raw_lines=ev_raw,
    ),
    index=Index(bbox=bbox, anomalies=1),
)

result = pymfx.validate(mfx)
print(result)
print(f"{len(points)} trajectory points, {len(event_list)} events")

## 1. GeoJSON

The trajectory is encoded as a GeoJSON `LineString` feature.
Events are exported as individual `Point` features with their metadata as properties.

In [ ]:
geojson_str = pymfx.to_geojson(mfx)
print(geojson_str[:800], "...")

In [ ]:
import json

geo = json.loads(geojson_str)
print("type            :", geo['type'])
print("feature count   :", len(geo['features']))
print("feature types   :", [f['geometry']['type'] for f in geo['features']])

In [ ]:
# Save to file
with open('/tmp/alpine_survey.geojson', 'w') as f:
    f.write(geojson_str)
print("Saved to /tmp/alpine_survey.geojson")

## 2. GPX

A standard GPX 1.1 `<trk>` with one `<trkpt>` per trajectory point.
Events are exported as `<wpt>` waypoints.

In [ ]:
gpx_str = pymfx.to_gpx(mfx)
print(gpx_str[:600], "...")

In [ ]:
# Save to file
with open('/tmp/alpine_survey.gpx', 'w') as f:
    f.write(gpx_str)
print("Saved to /tmp/alpine_survey.gpx")

## 3. KML

A KML `<Placemark>` containing a `<LineString>` with coordinates `lon,lat,alt`.

In [ ]:
kml_str = pymfx.to_kml(mfx)
print(kml_str[:600], "...")

In [ ]:
with open('/tmp/alpine_survey.kml', 'w') as f:
    f.write(kml_str)
print("Saved to /tmp/alpine_survey.kml")

## 4. CSV

A flat CSV with one row per trajectory point. All schema fields become columns.

In [ ]:
csv_str = pymfx.to_csv(mfx)

# Preview first few lines
lines = csv_str.strip().split('\n')
print('\n'.join(lines[:6]))
print(f"\n({len(lines) - 1} data rows + 1 header)")

In [ ]:
with open('/tmp/alpine_survey.csv', 'w') as f:
    f.write(csv_str)
print("Saved to /tmp/alpine_survey.csv")

## 5. CLI equivalent

Every conversion above has a one-liner CLI equivalent:

```bash
pymfx flight.mfx --export geojson
pymfx flight.mfx --export gpx  -o flight.gpx
pymfx flight.mfx --export kml  -o flight.kml
pymfx flight.mfx --export csv  -o flight.csv
```

In [ ]:
# Write the demo file so we can test the CLI
pymfx.write(mfx, '/tmp/alpine_survey.mfx')

import subprocess
result = subprocess.run(
    ['python', '-m', 'pymfx', '/tmp/alpine_survey.mfx', '--export', 'geojson'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout[:400], "...")

## Summary

| Function | Returns | Use case |
|---|---|---|
| `pymfx.to_geojson(mfx)` | `str` | Web maps, Leaflet, Mapbox |
| `pymfx.to_gpx(mfx)` | `str` | Garmin, Strava, QGIS |
| `pymfx.to_kml(mfx)` | `str` | Google Earth / Maps |
| `pymfx.to_csv(mfx)` | `str` | Spreadsheets, pandas read_csv |